# Unit 2 · Python and Colab for ML

Reproduce the 08:00 rental forecast for November 1, 2012. Start with Python names and functions, then open real records, filter rows, and read the shapes of the model inputs.

In Colab, click **Connect** if needed. Click the play button beside a code cell or press **Shift + Enter**. Run the cells from the top. The Python session remembers the names you define. To clear those names, restart the session from **Runtime** and run the cells from the top again. The notebook text stays.

The main exercise uses embedded, attributed teaching data. It needs pandas, NumPy, and IPython, which are available in Colab. It can run offline in a local Jupyter installation with those packages already installed.

[Official Colab notebook controls](https://colab.research.google.com/notebooks/basic_features_overview.ipynb)

## 1. Name the inputs

The original data has 456 earlier workday hours at 08:00. Across those hours, 214,195 rentals were recorded. A variable gives a value a name. Read `=` as “store this value under this name.” `print` displays the values supplied to it.

In [ ]:
rentals_total = 214195
hours_seen = 456
print(rentals_total, hours_seen)

A **list** holds several values in order. For example, these are the measured counts for 07:00, 08:00, and 09:00 on October 26. Python starts counting positions at zero, so `[0]` reads the first item. A `for` loop repeats the indented action for each item.

In [ ]:
october_26_counts = [421, 734, 403]
print("First value:", october_26_counts[0])
print("How many values?", len(october_26_counts))
for count in october_26_counts:
    print(count)

## 2. Define and call a function

A **function** gives a calculation a name. The definition lists the inputs it expects. The indented line does the work. `return` sends the result back to where the function was called. Divide total rentals by the number of matching hours and round to the nearest whole rental.

For these inputs, 214,195 ÷ 456 ≈ 469.73, which rounds to 470.

In [ ]:
def forecast(total, count):
    return round(total / count)

predicted = forecast(rentals_total, hours_seen)
print(predicted)

## 3. Read the observation and measure error

The count of 680 was recorded after 08:00 on November 1. Compute the **residual**, defined here as observed minus predicted. The positive sign means demand was 210 rentals higher than predicted.

In [ ]:
actual = 680
residual = actual - predicted
print(residual)

## 4. Read a CSV with pandas

`import` makes library tools available. `pd.read_csv` loads data into a **DataFrame**, or a table with named columns. The embedded text below contains twelve actual hours from the archive. `StringIO` lets `read_csv` read that text as it would read a file. For a CSV on disk, you would provide its path instead.

Each row gives the date, hour, workday flag, and recorded rental count. A workday flag of 1 means a workday. A flag of 0 means a weekend or holiday.

In [ ]:
from io import StringIO

import pandas as pd
from IPython.display import HTML, display

preview_csv = '''date,hour,workingday,rentals
2012-10-26,7,1,421
2012-10-26,8,1,734
2012-10-26,9,1,403
2012-10-27,7,0,86
2012-10-27,8,0,213
2012-10-27,9,0,328
2012-10-31,7,1,337
2012-10-31,8,1,621
2012-10-31,9,1,297
2012-11-01,7,1,420
2012-11-01,8,1,680
2012-11-01,9,1,331'''

rows = pd.read_csv(StringIO(preview_csv))
display(rows)

## 5. Filter rows and build groups

Use `.loc[condition]` to keep rows that pass a test. In the first line, `<` keeps earlier dates. In the second, `==` checks equality and `&` requires both flags. The two matching rows in this excerpt have 734 and 621 rentals.

`groupby` puts each pair of hour and workday values in its own group. `agg` calculates the count and the sum for each group. The last line reads the summary for workday 08:00 in the excerpt. This yields 2 and 1,355.

In [ ]:
cutoff = "2012-11-01"
history = rows.loc[rows["date"] < cutoff]
workday_eight = history.loc[(history["hour"] == 8) & (history["workingday"] == 1)]
display(workday_eight)

group_totals = history.groupby(["hour", "workingday"])["rentals"]
preview_summary = group_totals.agg(["count", "sum"])
display(preview_summary.loc[(8, 1)])

## 6. Forecast using the full historical groups

These saved aggregates come from all eligible rows before November 1, 2012 in the official archive. The short excerpt is for explaining the filter. The saved aggregates let us reproduce the original whole-rental forecasts offline. The last optional cell recomputes the aggregates from the complete archive.

On a workday, 07:00, 08:00, and 09:00 each have 456 historical observations. Apply the same function to each group.

In [ ]:
full_groups = pd.DataFrame({
    "hour": [7, 8, 9],
    "workingday": [1, 1, 1],
    "count": [456, 456, 456],
    "sum": [131628, 214195, 108510],
})
full_groups["prediction"] = [
    forecast(total, count) for total, count in zip(full_groups["sum"], full_groups["count"])
]
display(full_groups)

## 7. Check the shape of X and y

The three evaluation rows are on November 1. The model gets two inputs per row, hour and workday flag. The recorded rentals are the target used to check predictions.

A list of column names in brackets selects a table. One column name selects a series. `.to_numpy()` returns an array. Here `X.shape` is `(3, 2)` for three rows and two features. `y.shape` is `(3,)` for a one-dimensional array of three target values.

In [ ]:
morning = rows.loc[rows["date"] == cutoff].sort_values("hour")
X = morning[["hour", "workingday"]].to_numpy()
y = morning["rentals"].to_numpy()
print("X:", X)
print("y:", y)
print("Shapes:", X.shape, y.shape)

## 8. Plot forecast and recorded counts

Join each recorded count to the model output for the same hour and workday flag. Each pair of bars starts at zero and shares a maximum of 700 rentals. The labels also show the values directly. Read the 08:00 bars first.

In [ ]:
results = morning.merge(full_groups[["hour", "workingday", "prediction"]], on=["hour", "workingday"])
scale = 700
rows_html = []
for row in results.itertuples(index=False):
    rows_html.append(
        f'<tr><th scope="row">{row.hour:02}:00</th>'
        f'<td><meter min="0" max="{scale}" value="{row.prediction}"></meter> {row.prediction}</td>'
        f'<td><meter min="0" max="{scale}" value="{row.rentals}"></meter> {row.rentals}</td></tr>'
    )
display(HTML(f'<table><caption>Rentals per hour, each bar from 0 to {scale}</caption>'
             '<thead><tr><th scope="col">Hour</th><th scope="col">Forecast</th>'
             '<th scope="col">Recorded</th></tr></thead><tbody>'
             + "".join(rows_html) + '</tbody></table>'))

## Try it yourself · Predict 09:00

Find the 09:00 workday group in `full_groups`. Identify the historical count and sum. Use the `forecast` function again, find the actual rentals in `morning`, and calculate actual minus prediction. Predict the sign before revealing the answer.

<details>
<summary>Check your calculation</summary>

108,510 ÷ 456 ≈ 237.96, which rounds to 238. The recorded count was 331. The residual was 331 − 238 = +93 rentals, so the forecast was too low.

</details>

## Optional · Recompute the fixture from the official archive

Set `verify_archive = True` to run the verification. If `bike-sharing-dataset.zip` is in your notebook’s working directory, the code uses that file. Otherwise it downloads the official archive. It checks both the twelve displayed records and the three full-history group summaries.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile
from io import BytesIO

verify_archive = False
archive_path = Path("bike-sharing-dataset.zip")
archive_url = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"

if verify_archive:
    if archive_path.is_file():
        source_bytes = archive_path.read_bytes()
    else:
        with urlopen(archive_url, timeout=15) as response:
            source_bytes = response.read()
    with ZipFile(BytesIO(source_bytes)) as archive:
        original = pd.read_csv(archive.open("hour.csv"))
    original = original.rename(columns={"dteday": "date", "hr": "hour", "cnt": "rentals"})
    row_keys = ["date", "hour", "workingday"]
    checked = rows.merge(original[row_keys + ["rentals"]], on=row_keys, suffixes=("_lesson", "_archive"))
    assert len(checked) == len(rows)
    assert (checked["rentals_lesson"] == checked["rentals_archive"]).all()
    archive_history = original.loc[original["date"] < cutoff]
    totals = archive_history.groupby(["hour", "workingday"])["rentals"].agg(["count", "sum"])
    recorded_groups = totals.loc[[(7, 1), (8, 1), (9, 1)]].reset_index()
    assert recorded_groups.equals(full_groups[["hour", "workingday", "count", "sum"]])
    print("Verified 12 observed rows and 3 historical groups.")

## Source

Fanaee-T, H. (2013). Bike Sharing. UCI Machine Learning Repository. The fixture uses hourly counts of Capital Bikeshare rentals and the original working-day flag. The forecast is computed from pre-November 1, 2012 records. [Dataset and license](https://doi.org/10.24432/C5W894)